In [ ]:
using BifurcationKit, Plots, Parameters, NLsolve
const BK = BifurcationKit

In [ ]:
function S(x, k_0, x_0)
	return 1 ./ (1 .+ exp.(-k_0 .* (x .- x_0)))
end

# function S(x, k_0, x_0)
# 	return k_0 .* tanh.(x .- x_0)
# end

function Fsingle(x, pars)
	@unpack k_0, x_0, Iin, k = pars
	dx = (-x .+ k.* S(x, k_0, x_0) .+ Iin)
	return dx
end

In [ ]:
k_0 = 3.0
x_0 = 1.3
Iin = 0.
k = 3.
pars = (k_0=k_0, x_0=x_0, Iin=Iin, k=k)

x = range(0, 10, length=100)
plot(x, S(x, k_0, x_0), xlabel="x", ylabel="S(x)")

In [ ]:
plot(x, x .- k .* S(x, k_0, x_0))

In [ ]:
rfs(x, p) = (x1 = x[1]) # what is that?
x0 = [0.]
prob = BifurcationProblem(Fsingle, x0, pars, (@lens _.k), record_from_solution = rfs)

In [ ]:
opts = ContinuationPar(p_min = 0., p_max = 10., n_inversion = 4,
                       ds = 0.01, dsmin = 0.001, dsmax = 0.1, max_steps = 1000,
                       nev = 3)
# br = continuation(prob, PALC(), opts; normC = norminf, bothside = true)

# diagram = bifurcationdiagram(prob, br, 2, (args...) -> opts)
# plot(diagram, grid = false, legend = false)

In [ ]:
opts = ContinuationPar(p_min = 0., p_max = 10., n_inversion = 4,
                       ds = 0.01, dsmin = 0.001, dsmax = 0.1, max_steps = 1000,
                       nev = 3)
diagram = bifurcationdiagram(prob, PALC(), 2, (args...) -> opts)
plot(diagram, grid = false, legend = false)

In [ ]:
const I0 = 1e-12 # A
# const I0 = 1
const κ = 0.7
const Vdd = 1.8 # V
const UT= 25*1e-3 # V
const C=1e-3 # F

In [ ]:
# Static transistor equations
Icmp(Vin,V1) = I0 * exp(κ*(Vdd-Vin)/UT) * (1 - exp(-(Vdd-V1)/UT))
I1(V0,V1) = I0 * exp(κ*V0/UT) * (1 - exp(-V1/UT))
Ilin1(V1,Vout) = I0 * exp((κ*V1 - Vout)/UT) * (1 - exp(-(V1-Vout)/UT))
Ilin2(Vout,V2) = I0 * exp((κ*Vout - V2)/UT) * (1 - exp(-(Vout-V2)/UT))
Ilin3(Vlin,V2) = I0 * exp(κ*Vlin/UT) * (1 - exp(-V2/UT));

In [ ]:
function softmax(x, ϵ)

	tmp = [x;1]
	tmp2 = tmp.* exp.(tmp/ϵ) ./ (sum(exp.(tmp/ϵ)))

	# @show tmp
	# @show exp.(tmp/ϵ)

	return maximum(tmp2)[1]

end

In [ ]:
ϵ = 1e-2
icmp(Vin, v1, vout) = Icmp(Vin, log(softmax([v1, vout,1],ϵ)))
i1(V0, v1, vout) = I1(V0, log(softmax([v1, vout, 1],ϵ)))
ilin1(v1, vout, v2) = Ilin1(log(softmax([v1, vout, 1],ϵ)), log(softmax([vout, v2, 1],ϵ)))
ilin2(vout, v2) = Ilin2(log(softmax([vout, v2, 1],ϵ)), log(softmax([v2, 1],ϵ)))
ilin3(Vlin, v2) = Ilin3(Vlin, log(softmax([v2, 1],ϵ)))

In [ ]:
V_P_diode(I) = Vdd - UT/κ * log(I/I0)
V_N_diode(I) = UT/κ * log(I/I0);

In [ ]:
function Imode_sigmoid(x,pars)

	@unpack Iin, Ilin, I_0 = pars
    
    Vlin = V_N_diode.(Ilin)
    V0 = V_N_diode.(I_0)
    Vin = V_P_diode.(Iin)

    V1, Vout, V2 = x

    # print(x)

    [Icmp(Vin,x[1]) - I1(V0,x[1]) - Ilin1(x[1],x[2]),
    Ilin1(x[1],x[2]) - Ilin2(x[2],x[3]),
    Ilin2(x[2],x[3]) - Ilin3(Vlin,x[3])] ./C
    
end

In [ ]:
function Imode_sigmoid_log(x,pars)

	@unpack Iin, Ilin, I_0 = pars
    
    Vlin = V_N_diode.(Ilin)
    V0 = V_N_diode.(I_0)
    Vin = V_P_diode.(Iin)

	v1, vout, v2 = x

	[icmp(Vin,x[1], x[2]) - i1(V0,x[1], x[2]) - ilin1(x[1],x[2], x[3]),
    ilin1(x[1],x[2], x[3]) - ilin2(x[2],x[3]),
    ilin2(x[2],x[3]) - ilin3(Vlin,x[3])]

end

In [ ]:
Ilin = 100e-9 # A
I_0 = 300e-9 # A
Igain = 500e-9 # A
Iin = 500e-9 # A
Vgain = V_N_diode(Igain)
Vlin = V_N_diode.(Ilin)
V0 = V_N_diode.(I_0)
Vin = V_P_diode.(Iin)

pars = (Iin = Iin, Ilin = Ilin, I_0 = I_0)

x0=[1.7, 0.9, 0.3]

solNL = nlsolve(x -> Imode_sigmoid(x,pars), x0, iterations=convert(Int64,1e6), ftol=1e-9, xtol=1e-6)

# Iin = 1e-9 # A

In [ ]:
rfs(x, p) = (x1= x[1], x2 = x[2], x3=x[3], y=p)
prob = BifurcationProblem(Imode_sigmoid_log, solNL.zero, pars, (@lens _.Iin), record_from_solution = rfs)

In [ ]:
optnewton = NewtonPar(tol = 1e-16, max_iterations = 100, verbose=true)
opts = ContinuationPar(p_min = 1e-9, p_max = 600e-9, n_inversion = 50, ds = 1e-6, dsmin = 1e-12, dsmax = 1e-3, max_steps = 1000, nev = 3)
br = continuation(prob, PALC(), opts; normC = norminf, bothside = true)